# Agentic AI - Reflection in a Linkedin Post Workflow

**Modified by:** [Seyed Mohammadreza Hashemi](https://github.com/MrMrProgrammer/agentic-ai)

In [1]:
from utils.utils import show_output
import aisuite as ai

client = ai.Client()

### 📝 `generate_draft` Function

In [8]:
def generate_draft(
    prompt: str,
    tone: str,
    model: str = "openai:gpt-5-nano"
) -> str:
    instruction = f"""
    شما یک نویسنده حرفه‌ای محتوای فارسی برای LinkedIn هستید.

    وظیفه:
    بر اساس موضوع و لحن مشخص‌شده، یک پست LinkedIn فارسی تولید کن.

    موضوع:
    {prompt}

    لحن موردنظر:
    {tone}

    الزامات محتوا:
    - متن باید فارسی و مناسب انتشار در LinkedIn باشد.
    - لحن متن باید دقیقاً با لحن موردنظر هماهنگ باشد.
    - متن باید یک ایده یا پیام اصلی مشخص داشته باشد.
    - متن باید طبیعی و انسانی به نظر برسد.
    - از مقدمه‌چینی طولانی و جملات کلیشه‌ای خودداری کن.
    - متن باید خوانایی مناسبی برای فضای LinkedIn داشته باشد.
    - در صورت نیاز، پاراگراف‌ها را کوتاه نگه دار.
    - از Markdown پیچیده، جدول و HTML استفاده نکن.
    - هیچ توضیحی درباره فرآیند تولید ارائه نده.
    - خروجی فقط شامل متن اصلی پست باشد.

    محدودیت‌ها:
    - طول کل پست بین 300 تا 500 کاراکتر باشد.
    - تعداد هشتگ‌ها بین 3 تا 5 عدد باشد.
    - هشتگ‌ها نیز جزو تعداد کل کاراکترهای پست محسوب می‌شوند.

    خروجی:
    فقط متن پست را برگردان.
    """

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": instruction
            }
        ],
        temperature=1.0,
    )

    return response.choices[0].message.content

### 🔍 `reflect_on_draft` Function

In [18]:
def check_post_quality(
   post_text: str,
   max_length=500,
   min_length=300,
   min_hashtags=3,
   max_hashtags=5
):
   length = len(post_text)

   within_length_limit = min_length <= length <= max_length

   words = post_text.split(" ")
   hashtag_count = 0

   for word in words:
      if word.startswith("#"):
         hashtag_count += 1

   within_hashtag_range = True if (hashtag_count >= min_hashtags and hashtag_count <= max_hashtags) else False
   
   return {
      "length": length,
      "within_length_limit": within_length_limit,
      "hashtag_count": hashtag_count,
      "within_hashtag_range": within_hashtag_range
   }

In [10]:
def reflect_on_draft(
    draft: str,
    quality_signals: dict,
    model: str = "openai:gpt-5.6-luna",
) -> str:
    instruction = f"""
    شما یک منتقد حرفه‌ای محتوای فارسی برای LinkedIn هستید.

    وظیفه:
    Draft زیر را به‌صورت انتقادی بررسی کن و مشکلات آن را شناسایی کن.
    نتیجه‌ی واقعی بررسی کیفیت Draft نیز در اختیار تو قرار گرفته است.
    در نقد خود حتماً این quality signals را در نظر بگیر.

    Draft:
    {draft}

    Quality Signals:
    {quality_signals}

    بر اساس Draft و Quality Signals:
    - مشکلات مربوط به طول پست را بررسی کن.
    - تعداد هشتگ‌ها را بررسی کن.
    - مواردی که محدودیت‌های تعیین‌شده رعایت نشده‌اند را به‌صورت صریح بیان کن.
    - کیفیت محتوا، وضوح پیام، ساختار، خوانایی و تناسب با LinkedIn را بررسی کن.
    - نقد باید مشخص و قابل استفاده برای مرحله‌ی بعدی اصلاح Draft باشد.
    - برای هر مشکل مهم، توضیح بده که چه چیزی باید اصلاح شود.
    - اگر یک محدودیت رعایت شده است، آن را به‌عنوان مشکل مطرح نکن.
    - فقط بر اساس Draft و Quality Signals قضاوت کن و اطلاعاتی را حدس نزن.

    خروجی:
    یک نقد ساختاریافته و قابل استفاده برای اصلاح Draft ارائه بده.
    """

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": instruction}
        ],
        temperature=1.0,
    )

    return response.choices[0].message.content

### 🔁 `revise_draft` Function

In [11]:
def revise_draft(
    original_draft: str,
    reflection: str,
    model: str = "openai:gpt-5.6-luna",
) -> str:
    instruction = f"""
    شما یک نویسنده حرفه‌ای محتوای فارسی برای LinkedIn هستید.

    وظیفه:
    Draft زیر را بر اساس نقد و بازخورد ارائه‌شده اصلاح کن.

    Original Draft:
    {original_draft}

    Reflection:
    {reflection}

    الزامات اصلاح:
    - مشکلات و ایرادهای مطرح‌شده در Reflection را برطرف کن.
    - پیام اصلی Draft را حفظ کن، مگر اینکه Reflection مشخصاً نیاز به تغییر آن را مطرح کرده باشد.
    - متن باید طبیعی، روان و مناسب انتشار در LinkedIn باشد.
    - لحن و سبک کلی متن را حفظ کن.
    - از جملات کلیشه‌ای و توضیحات غیرضروری خودداری کن.
    - متن را واضح و خوانا نگه دار.
    - از Markdown پیچیده، جدول و HTML استفاده نکن.

    محدودیت‌های قابل‌سنجش:
    - طول کل پست حداکثر 500 کاراکتر باشد.
    - تعداد هشتگ‌ها بین 3 تا 5 عدد باشد.
    - هشتگ‌ها نیز جزو تعداد کل کاراکترهای پست محسوب می‌شوند.

    خروجی:
    فقط نسخه اصلاح‌شده پست را برگردان.
    هیچ توضیح، تحلیل یا مقدمه‌ای خارج از متن پست ارائه نده.
    """

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": instruction}
        ],
        temperature=1.0,
    )

    return response.choices[0].message.content

### 🧪 Test the Reflective Writing Workflow

## تست اول

In [15]:
def run_linkedin_workflow(topic, tone="حرفه‌ای", model="openai:gpt-4o-mini"):

    # Agente 1 – Draft
    draft = generate_draft(
        prompt=topic,
        tone=tone
    )
    show_output(
        "مرحله اول: پیش‌نویس",
        draft,
        background="#fff8dc",
        text_color="#333333"
    )

    # # Agente 2 – Reflection
    quality_signals = check_post_quality(draft)
    feedback = reflect_on_draft(draft, quality_signals)
    show_output(
        "مرحله دوم – Reflection",
        feedback,
        background="#e0f7fa",
        text_color="#222222"
    )

    # # Agente 3 – Revision
    revised = revise_draft(draft, feedback)
    show_output(
        "مرحله سوم – بازنگری یا Revision",
        revised,
        background="#f3e5f5",
        text_color="#222222"
    )

In [16]:
topic = "من به تازگی در شرکت Bijak به عنوان Backend Developer استخدام شده ام."
tone = "دوستانه"

run_linkedin_workflow(
    topic=topic,
    tone=tone
)

## تست دوم

In [19]:
topic = "به تازگی دوره آموزشی Agentic AI رو درون سایت مکتب خونه که توسط استاد علیرضا اخوان‌پور ایجاد شده است را تمام کرده ام."
tone = "حرفه‌ای"

run_linkedin_workflow(
    topic=topic,
    tone=tone
)